# Multimodal Deepfake Detection for Kaggle: Video Swin Tiny + WavLM-Base+ + Cross-Attention Fusion

This notebook is the Kaggle-compatible version of the new research approach. It starts with a per-speaker pilot and keeps a clean path toward full-dataset training.

Training plan:
- Stage A: frozen unimodal training
- Stage B: partial unfreezing
- Stage C: fusion training with partially frozen encoders
- Stage D: end-to-end low-learning-rate fine-tuning

In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import random
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import FileLink, display
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.amp.autocast_mode import autocast
from torch.amp.grad_scaler import GradScaler
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import subprocess
import tempfile

try:
    import timm
except ImportError:
    timm = None

try:
    from transformers import AutoFeatureExtractor, WavLMModel
except ImportError:
    AutoFeatureExtractor = None
    WavLMModel = None

try:
    import torchaudio
except Exception:
    torchaudio = None

try:
    import scipy.io.wavfile as scipy_wavfile
except Exception:
    scipy_wavfile = None

print('Imports ready')

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_project_root() -> Path:
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return Path('/kaggle/working')
    return Path(r'F:\Sixth Semester\DEEPfake Papers\FakeBDTeen')


def detect_dataset_root() -> Path:
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates = [p for p in kaggle_input.iterdir() if p.is_dir()]
        for candidate in candidates:
            if list(candidate.rglob('*.mp4')):
                return candidate
    return resolve_project_root() / 'FakeBDTeen'


PROJECT_ROOT = resolve_project_root()
RAW_DATASET_ROOT = detect_dataset_root()
WORK_DIR = PROJECT_ROOT / 'outputs_video_swin_wavlm_kaggle'
WORK_DIR.mkdir(parents=True, exist_ok=True)

PILOT_CLIPS = 960
FULL_DATASET_MODE = False
RUN_TRAINING = False
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CONFIG = {
    'mode': 'pilot' if not FULL_DATASET_MODE else 'full',
    'pilot_clips': PILOT_CLIPS,
    'full_dataset_enabled': FULL_DATASET_MODE,
    'seed': SEED,
    'num_frames': 16,
    'frame_size': 224,
    'audio_sample_rate': 16000,
    'batch_size': 4,
    'num_workers': 2,
    'accumulation_steps': 4,
    'epochs_stage_0': 2,
    'epochs_stage_a': 4,
    'epochs_stage_b': 3,
    'epochs_stage_c': 5,
    'epochs_stage_d': 3,
    'lr_stage_a': 1e-4,
    'lr_stage_b': 8e-5,
    'lr_stage_c': 5e-5,
    'lr_stage_d': 1e-5,
    'weight_decay': 1e-4,
    'use_amp': torch.cuda.is_available(),
    'resume_from_checkpoints': True,
    'checkpoint_every_epoch': True,
    'aux_loss_weight': 0.3,
    'modality_drop_p': 0.1,
    'mixup_alpha': 0.4,
    'use_ema': True,
    'ema_decay': 0.999,
    'use_feature_cache': False,
    'dry_run': False,
    'run_stage_0': False,
    'held_out_speakers': [],
    'class_names': ['FF', 'FR', 'RF', 'RR'],
    # 2D Swin Tiny applied per-frame with learned temporal attention pooling.
    'video_model_name': 'swin_tiny_patch4_window7_224',
    'audio_model_name': 'microsoft/wavlm-base-plus',
}

set_seed(SEED)
print('Project root:', PROJECT_ROOT)
print('Dataset root:', RAW_DATASET_ROOT)
print('Working directory:', WORK_DIR)
print('Device:', DEVICE)

## 1. Import Required Libraries and Dependencies

This notebook uses PyTorch, timm, transformers, OpenCV, NumPy, pandas, and scikit-learn.

## 2. Set Up Configuration and Hyperparameters

The notebook keeps a pilot/full switch so the same structure can scale later.

In [ ]:
def normalize_path_text(text: str) -> str:
    # normalize underscores and spacing, then map known labels to a canonical form
    s = text.replace('_', ' ')
    s = s.replace('+', ' + ')
    s = ' '.join(s.split())
    lower = s.lower()
    if 'fake video' in lower:
        s = s.replace(s[s.lower().find('fake video'):s.lower().find('fake video') + len('fake video')], 'Fake Video')
    if 'real video' in lower:
        s = s.replace(s[s.lower().find('real video'):s.lower().find('real video') + len('real video')], 'Real Video')
    if 'fake audio' in lower:
        s = s.replace(s[s.lower().find('fake audio'):s.lower().find('fake audio') + len('fake audio')], 'Fake Audio')
    if 'real audio' in lower:
        s = s.replace(s[s.lower().find('real audio'):s.lower().find('real audio') + len('real audio')], 'Real Audio')
    return ' '.join(s.split())


def infer_class_name(path_text: str) -> str:
    normalized = normalize_path_text(path_text)
    if 'Fake Video + Fake Audio' in normalized:
        return 'FF'
    if 'Fake Video + Real Audio' in normalized:
        return 'FR'
    if 'Real Video + Fake Audio' in normalized:
        return 'RF'
    if 'Real Video + Real Audio' in normalized:
        return 'RR'
    raise ValueError(f'Unable to infer class from path: {path_text}')


def infer_gender_from_path(path_text: str) -> str:
    if 'Boys' in path_text:
        return 'Boys'
    if 'Girls' in path_text:
        return 'Girls'
    return 'Unknown'


def infer_language_from_path(path_text: str) -> str:
    if 'Bangla' in path_text:
        return 'Bangla'
    if 'English' in path_text:
        return 'English'
    return 'Unknown'


def discover_clip_records(root: Path) -> List[Dict[str, str]]:
    records: List[Dict[str, str]] = []
    for mp4_path in root.rglob('*.mp4'):
        relative_path = mp4_path.relative_to(root)
        normalized_relative = Path(*[normalize_path_text(part) for part in relative_path.parts])
        rel_text = str(normalized_relative)
        records.append({
            'path': str(mp4_path),
            'relative_path': rel_text,
            'speaker_id': normalized_relative.parts[-2] if len(normalized_relative.parts) >= 2 else 'unknown',
            'label': infer_class_name(rel_text),
            'gender': infer_gender_from_path(rel_text),
            'language': infer_language_from_path(rel_text),
        })
    return records


def verify_dataset_balance(records: Sequence[Dict[str, str]]) -> None:
    if not records:
        print('No records found for balance verification.')
        return
    frame = pd.DataFrame(records)
    ctab = pd.crosstab(
        index=frame['label'],
        columns=[frame['gender'], frame['language']],
        dropna=False,
    )
    print('Label x Gender x Language crosstab:')
    print(ctab)

    nonzero = ctab.to_numpy().astype(float)
    nonzero = nonzero[nonzero > 0]
    if nonzero.size == 0:
        print('No non-zero cells for deviation check.')
        return
    expected = float(nonzero.mean())
    lower = expected * 0.8
    upper = expected * 1.2
    flagged = []
    for row_label in ctab.index:
        for col_label in ctab.columns:
            val = float(ctab.loc[row_label, col_label])
            if val > 0 and (val < lower or val > upper):
                flagged.append((row_label, col_label, int(val)))
    if flagged:
        print('WARNING: imbalance cells (>20% deviation from expected):')
        for row_label, col_label, val in flagged:
            print(f'  label={row_label} gender={col_label[0]} language={col_label[1]} count={val}')
    else:
        print('Balance check passed within 20% tolerance.')


def reserve_held_out_speakers(records: Sequence[Dict[str, str]], n: int = 3, seed: int = SEED) -> Tuple[List[Dict[str, str]], List[Dict[str, str]], List[str]]:
    rng = random.Random(seed)
    speaker_to_records: Dict[str, List[Dict[str, str]]] = defaultdict(list)
    for rec in records:
        speaker_to_records[rec['speaker_id']].append(rec)

    boys = [sid for sid, recs in speaker_to_records.items() if any(r.get('gender') == 'Boys' for r in recs)]
    girls = [sid for sid, recs in speaker_to_records.items() if any(r.get('gender') == 'Girls' for r in recs)]
    rng.shuffle(boys)
    rng.shuffle(girls)

    selected: List[str] = []
    if boys:
        selected.append(boys.pop())
    if girls and len(selected) < n:
        selected.append(girls.pop())

    remaining = [sid for sid in speaker_to_records.keys() if sid not in selected]
    rng.shuffle(remaining)
    while len(selected) < n and remaining:
        selected.append(remaining.pop())

    selected = selected[:n]
    held_out = [rec for rec in records if rec['speaker_id'] in selected]
    kept = [rec for rec in records if rec['speaker_id'] not in selected]

    with (WORK_DIR / 'held_out_manifest.json').open('w', encoding='utf-8') as f:
        json.dump(held_out, f, indent=2)

    return kept, held_out, selected


all_records_full = discover_clip_records(RAW_DATASET_ROOT)
print('Total clips discovered:', len(all_records_full))
print('Unique speakers:', len({record['speaker_id'] for record in all_records_full}))

verify_dataset_balance(all_records_full)

all_records, held_out_records, held_out_speakers = reserve_held_out_speakers(all_records_full, n=3, seed=SEED)
CONFIG['held_out_speakers'] = held_out_speakers
print('Reserved held-out speakers:', held_out_speakers)
print('Held-out clips:', len(held_out_records))
print('Trainable clips after hold-out reservation:', len(all_records))

## 3. Load and Preprocess Dataset (970 Clips)

The pilot subset is created from speaker groups so the same speaker does not leak across splits.

In [ ]:
def build_pilot_per_speaker(records: Sequence[Dict[str, str]], per_speaker: int = 4, seed: int = 42) -> List[Dict[str, str]]:
    """Select `per_speaker` clips from each speaker folder using record['speaker_id'].
    If a speaker has fewer than `per_speaker` clips, take all available clips from that speaker.
    Saves a manifest named `pilot_manifest_per_speaker_{per_speaker}.json` in WORK_DIR.
    """
    rng = random.Random(seed)
    groups: Dict[str, List[Dict[str, str]]] = defaultdict(list)
    for record in records:
        speaker_key = record['speaker_id'] or 'unknown'
        groups[speaker_key].append(record)

    selected: List[Dict[str, str]] = []
    for items in groups.values():
        rng.shuffle(items)
        take = min(per_speaker, len(items))
        selected.extend(items[:take])

    manifest_path = WORK_DIR / f'pilot_manifest_per_speaker_{per_speaker}.json'
    with manifest_path.open('w', encoding='utf-8') as f:
        json.dump(selected, f, indent=2)
    print('Saved per-speaker pilot manifest to', manifest_path)
    print('Pilot clips selected (total):', len(selected), 'speakers:', len(groups))
    print('Pilot class distribution:', Counter(item['label'] for item in selected))
    return selected


# Build pilot by taking 4 clips per speaker from non-held-out records
pilot_records = build_pilot_per_speaker(all_records, per_speaker=4, seed=SEED)
print('Pilot speakers:', len({item['speaker_id'] for item in pilot_records}))

## 4. Implement Video Preprocessing Pipeline

This section defines frame extraction, resizing, normalization, and augmentation hooks.

In [ ]:
def extract_video_frames(video_path: str, num_frames: int = 8, size: int = 224) -> torch.Tensor:
    capture = cv2.VideoCapture(video_path)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        capture.release()
        return torch.zeros(num_frames, 3, size, size)
    frame_indices = np.linspace(0, frame_count - 1, num_frames).astype(int)
    frames = []
    for frame_index in frame_indices:
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = capture.read()
        if not ok:
            frame = np.zeros((size, size, 3), dtype=np.uint8)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (size, size), interpolation=cv2.INTER_AREA)
        frame = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
        frames.append(frame)
    capture.release()
    stacked = torch.stack(frames, dim=0)
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=stacked.dtype, device=stacked.device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], dtype=stacked.dtype, device=stacked.device).view(1, 3, 1, 1)
    stacked = (stacked - mean) / std
    return stacked


def preprocess_video_clip(video_path: str, train_mode: bool = True) -> torch.Tensor:
    frames = extract_video_frames(video_path, num_frames=CONFIG['num_frames'], size=CONFIG['frame_size'])
    augment_fn: Optional[Callable[[torch.Tensor], torch.Tensor]] = globals().get('apply_video_augmentations')
    if train_mode and augment_fn is not None:
        frames = augment_fn(frames)
    return frames

## 5. Implement Audio Preprocessing Pipeline

This section defines waveform loading and lightweight audio augmentation hooks for WavLM.

In [ ]:
def load_audio_waveform(video_path: str, sample_rate: int = 16000, duration: float = 10.0) -> torch.Tensor:
    """Extract audio from a video using ffmpeg then load with torchaudio or scipy fallback.
    Returns a 1 x N torch.FloatTensor sampled at `sample_rate` and trimmed/padded to `duration` seconds.
    """
    tmp_file = Path(tempfile.NamedTemporaryFile(suffix='.wav', delete=False).name)
    try:
        cmd = [
            'ffmpeg', '-y', '-i', str(video_path),
            '-ar', str(sample_rate), '-ac', '1', '-t', str(duration), str(tmp_file)
        ]
        try:
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        except Exception as e:
            print('ffmpeg extraction failed, returning zeros:', e)
            return torch.zeros(1, int(sample_rate * duration))

        waveform = None
        if torchaudio is not None:
            try:
                wav, sr = torchaudio.load(str(tmp_file))
                if sr != sample_rate:
                    wav = torchaudio.transforms.Resample(sr, sample_rate)(wav)
                waveform = wav
            except Exception:
                waveform = None
        if waveform is None and scipy_wavfile is not None:
            try:
                sr, data = scipy_wavfile.read(str(tmp_file))
                data = np.asarray(data, dtype=np.float32)
                if data.ndim > 1:
                    data = data.mean(axis=1)
                # normalize if integers
                if data.dtype != np.float32:
                    maxv = max(1.0, np.max(np.abs(data)))
                    data = data.astype(np.float32) / maxv
                waveform = torch.from_numpy(data).unsqueeze(0)
                if sr != sample_rate:
                    # simple resample via linear interpolation
                    new_len = int(round(waveform.size(1) * (sample_rate / sr)))
                    interp = np.interp(np.linspace(0, waveform.size(1) - 1, new_len), np.arange(waveform.size(1)), waveform.numpy().squeeze())
                    waveform = torch.from_numpy(interp.astype(np.float32)).unsqueeze(0)
            except Exception:
                waveform = None

        if waveform is None:
            waveform = torch.zeros(1, int(sample_rate * duration))

        expected = int(sample_rate * duration)
        if waveform.size(1) < expected:
            waveform = F.pad(waveform, (0, expected - waveform.size(1)))
        elif waveform.size(1) > expected:
            waveform = waveform[:, :expected]
        return waveform
    finally:
        tmp_file.unlink(missing_ok=True)


def preprocess_audio_clip(video_path: str, train_mode: bool = True) -> torch.Tensor:
    waveform = load_audio_waveform(video_path, sample_rate=CONFIG['audio_sample_rate'])
    if train_mode:
        waveform = apply_audio_augmentations(waveform)
    return waveform

# Note: `apply_audio_augmentations` is defined later in the augmentations cell
# to avoid duplicate definitions across notebook cells. The call above will
# resolve to that single definition when the augmentation cell is executed.


print('Audio preprocessing ready (torchaudio={} scipy={})'.format(torchaudio is not None, scipy_wavfile is not None))

## 6. Build Video Swin Tiny Backbone

The notebook prefers a Swin Tiny family model from timm.

In [ ]:
def resolve_video_swin_model_name() -> str:
    if timm is None:
        raise ImportError('timm is required for the video backbone.')
    video_models = timm.list_models('*video*swin*tiny*')
    if video_models:
        return video_models[0]
    swin_models = timm.list_models('*swin*tiny*')
    if swin_models:
        return swin_models[0]
    return 'swin_tiny_patch4_window7_224'


class SwinTinyPerFrameBackbone(nn.Module):
    """2D Swin Tiny applied per-frame with learned temporal attention pooling. Not the Video Swin Transformer."""
    def __init__(self, pretrained: bool = True):
        super().__init__()
        if timm is None:
            raise ImportError('timm is required for the video backbone.')
        self.model_name = resolve_video_swin_model_name()
        self.model = timm.create_model(self.model_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        self.sequence_output = True

        # Infer the actual backbone feature size because some timm Swin variants
        # report a stale num_features value even though the forward output is smaller.
        self.encoder_dim = self._infer_feature_dim()

        # Temporal fusion layers should be initialized here (not in forward)
        # so they are tracked by optimizer and saved in checkpoints.
        self.temporal_dim = max(64, self.encoder_dim // 2)
        self.feature_dim = self.temporal_dim
        self.temporal_proj = nn.Linear(self.encoder_dim, self.temporal_dim)
        self.temporal_attn = nn.MultiheadAttention(self.temporal_dim, num_heads=4, batch_first=True)
        self.temporal_ln = nn.LayerNorm(self.temporal_dim)

    def _infer_feature_dim(self) -> int:
        try:
            if hasattr(self.model, 'num_features') and isinstance(self.model.num_features, int):
                candidate = int(self.model.num_features)
            else:
                candidate = None
            with torch.no_grad():
                dummy = torch.zeros(1, 3, CONFIG['frame_size'], CONFIG['frame_size'])
                was_training = self.model.training
                self.model.eval()
                output = self.model(dummy)
                if was_training:
                    self.model.train()
            if isinstance(output, torch.Tensor) and output.ndim >= 2:
                inferred = int(output.shape[-1])
                if candidate is None or candidate != inferred:
                    return inferred
                return candidate
            if candidate is not None:
                return candidate
        except Exception:
            pass
        return 384

    def forward(self, frames: torch.Tensor) -> torch.Tensor:
        # Expect frames: [B, T, C, H, W]
        batch_size, num_frames, channels, height, width = frames.shape

        # Per-frame Swin encoding
        frames = frames.view(batch_size * num_frames, channels, height, width)
        per_frame = self.model(frames)  # [B*T, F]
        per_frame = per_frame.view(batch_size, num_frames, -1)  # [B, T, F]

        # Temporal attention pooling across frame tokens
        proj = self.temporal_proj(per_frame)  # [B, T, D]
        attn_out, _ = self.temporal_attn(proj, proj, proj)
        return self.temporal_ln(attn_out)


print('Video Swin Tiny backbone ready')

## 7. Build WavLM-Base+ Audio Backbone

The audio backbone uses WavLM-Base+ for robust speech embeddings.

In [ ]:
class WavLMBasePlusBackbone(nn.Module):
    def __init__(self, model_name: str = 'microsoft/wavlm-base-plus'):
        super().__init__()
        if WavLMModel is None or AutoFeatureExtractor is None:
            raise ImportError('transformers is required for WavLM-Base+ feature extraction.')
        self.feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
        self.model = WavLMModel.from_pretrained(model_name)
        self.feature_dim = self.model.config.hidden_size
        self.sequence_output = True

    def forward(self, waveform: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        waveform_np = waveform.detach().cpu().numpy()
        features = self.feature_extractor(
            waveform_np,
            sampling_rate=16000,
            return_tensors='pt',
            padding=True,
        )
        input_values = features['input_values'].to(waveform.device)
        feature_attention_mask = features.get('attention_mask')
        if feature_attention_mask is not None:
            feature_attention_mask = feature_attention_mask.to(waveform.device)
        outputs = self.model(input_values=input_values, attention_mask=feature_attention_mask)
        return outputs.last_hidden_state


print('WavLM-Base+ backbone ready')

## 8. Implement Cross-Attention Fusion Module

Cross-attention fuses video and audio embeddings before classification.

In [ ]:
class CrossAttentionFusion(nn.Module):
    def __init__(self, video_dim: int, audio_dim: int, fusion_dim: int = 512, num_heads: int = 8):
        super().__init__()
        self.video_proj = nn.Linear(video_dim, fusion_dim)
        self.audio_proj = nn.Linear(audio_dim, fusion_dim)
        self.cross_attn_video = nn.MultiheadAttention(fusion_dim, num_heads, batch_first=True)
        self.cross_attn_audio = nn.MultiheadAttention(fusion_dim, num_heads, batch_first=True)
        self.norm_video = nn.LayerNorm(fusion_dim)
        self.norm_audio = nn.LayerNorm(fusion_dim)
        self.out_proj = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.return_weights = False

    def forward(self, video_seq: torch.Tensor, audio_seq: torch.Tensor):
        video_proj = self.video_proj(video_seq)
        audio_proj = self.audio_proj(audio_seq)
        video_ctx, video_attn_weights = self.cross_attn_video(video_proj, audio_proj, audio_proj)
        audio_ctx, audio_attn_weights = self.cross_attn_audio(audio_proj, video_proj, video_proj)
        video_pool = self.norm_video(video_ctx).mean(dim=1)
        audio_pool = self.norm_audio(audio_ctx).mean(dim=1)
        fused = torch.cat([video_pool, audio_pool], dim=-1)
        fused_output = self.out_proj(fused)
        if self.return_weights:
            return fused_output, video_attn_weights, audio_attn_weights
        return fused_output


print('Cross-attention fusion ready')

## 9. Construct Full Multimodal Model

This combines the two backbones, fusion block, and final classifier.

In [ ]:
class MultimodalDeepfakeModel(nn.Module):
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.video_backbone = SwinTinyPerFrameBackbone(pretrained=True)
        self.audio_backbone = WavLMBasePlusBackbone()
        self.fusion = CrossAttentionFusion(self.video_backbone.feature_dim, self.audio_backbone.feature_dim)
        self.video_aux_head = nn.Linear(self.video_backbone.feature_dim, num_classes)
        self.audio_aux_head = nn.Linear(self.audio_backbone.feature_dim, num_classes)
        self.attn_gate = nn.Linear(512, 512)
        self.classifier = nn.Linear(512, num_classes)

    def extract_features(self, video_frames: torch.Tensor, audio_waveform: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        video_features = self.video_backbone(video_frames)
        audio_features = self.audio_backbone(audio_waveform, attention_mask=attention_mask)
        return video_features, audio_features

    def compute_aux_logits(self, video_features: torch.Tensor, audio_features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        video_aux_logits = self.video_aux_head(video_features.mean(dim=1))
        audio_aux_logits = self.audio_aux_head(audio_features.mean(dim=1))
        return video_aux_logits, audio_aux_logits

    def apply_modality_dropout(self, video_features: torch.Tensor, audio_features: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if not self.training:
            return video_features, audio_features
        p = float(CONFIG.get('modality_drop_p', 0.1))
        dropped_video = False
        if random.random() < p:
            video_features = torch.zeros_like(video_features)
            dropped_video = True
        if not dropped_video and random.random() < p:
            audio_features = torch.zeros_like(audio_features)
        return video_features, audio_features

    def fuse_logits(self, video_features: torch.Tensor, audio_features: torch.Tensor):
        fused = self.fusion(video_features, audio_features)
        if isinstance(fused, tuple):
            fused_vec, video_w, audio_w = fused
        else:
            fused_vec = fused
            video_w, audio_w = None, None
        weights = torch.sigmoid(self.attn_gate(fused_vec))
        pooled = weights * fused_vec
        logits = self.classifier(pooled)
        if video_w is not None and audio_w is not None:
            return logits, video_w, audio_w
        return logits

    def forward(self, video_frames: torch.Tensor, audio_waveform: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        video_features, audio_features = self.extract_features(video_frames, audio_waveform, attention_mask=attention_mask)
        video_aux_logits, audio_aux_logits = self.compute_aux_logits(video_features, audio_features)
        video_features, audio_features = self.apply_modality_dropout(video_features, audio_features)
        fusion_out = self.fuse_logits(video_features, audio_features)
        if isinstance(fusion_out, tuple) and len(fusion_out) == 3:
            fusion_logits, video_attn_weights, audio_attn_weights = fusion_out
            return {
                'fusion': fusion_logits,
                'video_aux': video_aux_logits,
                'audio_aux': audio_aux_logits,
                'video_attn_weights': video_attn_weights,
                'audio_attn_weights': audio_attn_weights,
            }
        return {'fusion': fusion_out, 'video_aux': video_aux_logits, 'audio_aux': audio_aux_logits}


model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names'])).to(DEVICE)
print(model.__class__.__name__)

## 10. Implement Data Augmentation Strategies

This notebook keeps augmentation hooks lightweight and easy to extend.

In [ ]:
def apply_video_augmentations(frames: torch.Tensor) -> torch.Tensor:
    if frames.size(0) > 1 and random.random() < 0.5:
        frames = torch.flip(frames, dims=[3])
    if random.random() < 0.5:
        brightness_scale = random.uniform(0.9, 1.1)
        frames = torch.clamp(frames * brightness_scale, 0.0, 1.0)
    if random.random() < 0.3:
        frames = torch.clamp(frames + 0.01 * torch.randn_like(frames), 0.0, 1.0)
    return frames


def apply_audio_augmentations(waveform: torch.Tensor) -> torch.Tensor:
    if random.random() < 0.5:
        waveform = waveform + 0.001 * torch.randn_like(waveform)
    if random.random() < 0.3:
        shift = int(0.1 * CONFIG['audio_sample_rate'])
        roll = random.randint(-shift, shift)
        waveform = torch.roll(waveform, shifts=roll, dims=1)
    return waveform


print('Augmentation hooks ready')

## 11. Create Stratified DataLoader with Class-Balanced Sampling

The split is speaker-aware and the sampler balances the weaker classes.

In [ ]:
CLASS_TO_INDEX = {name: idx for idx, name in enumerate(CONFIG['class_names'])}


def build_class_weight_tensor(records: Sequence[Dict[str, str]], class_names: Sequence[str]) -> torch.Tensor:
    """Create inverse-frequency class weights for CrossEntropyLoss."""
    label_counts = Counter(record['label'] for record in records)
    total = max(1, len(records))
    num_classes = max(1, len(class_names))
    weights: List[float] = []
    for class_name in class_names:
        class_count = label_counts.get(class_name, 0)
        if class_count == 0:
            weights.append(0.0)
        else:
            weights.append(total / (num_classes * class_count))
    return torch.tensor(weights, dtype=torch.float32)


class DeepfakeDataset(Dataset):
    def __init__(self, records: Sequence[Dict[str, str]], train_mode: bool = True):
        self.records = list(records)
        self.train_mode = train_mode

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        record = self.records[idx]
        video_frames = preprocess_video_clip(record['path'], train_mode=self.train_mode)
        audio_waveform = preprocess_audio_clip(record['path'], train_mode=self.train_mode)
        label = CLASS_TO_INDEX[record['label']]
        return {'video_frames': video_frames, 'audio_waveform': audio_waveform, 'label': label, 'speaker_id': record['speaker_id'], 'path': record['path'], 'gender': record.get('gender', 'Unknown'), 'language': record.get('language', 'Unknown')}


class CachedDeepfakeDataset(Dataset):
    def __init__(self, records: Sequence[Dict[str, str]], cache_dir: Path):
        self.records = list(records)
        self.cache_dir = cache_dir

    def __len__(self) -> int:
        return len(self.records)

    def _record_hash(self, record: Dict[str, str]) -> str:
        return hashlib.md5(record['path'].encode('utf-8')).hexdigest()[:12]

    def __getitem__(self, idx: int):
        record = self.records[idx]
        rh = self._record_hash(record)
        video_path = self.cache_dir / f'{rh}_video.pt'
        audio_path = self.cache_dir / f'{rh}_audio.pt'
        video_features = torch.load(str(video_path), map_location='cpu')
        audio_features = torch.load(str(audio_path), map_location='cpu')
        label = CLASS_TO_INDEX[record['label']]
        return {'video_features': video_features, 'audio_features': audio_features, 'label': label, 'speaker_id': record['speaker_id'], 'path': record['path'], 'gender': record.get('gender', 'Unknown'), 'language': record.get('language', 'Unknown')}


def precompute_and_cache_features(records: Sequence[Dict[str, str]], cache_dir: Path, device: torch.device) -> None:
    cache_dir.mkdir(parents=True, exist_ok=True)
    probe_model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names'])).to(device)
    probe_model.eval()
    for rec in records:
        rh = hashlib.md5(rec['path'].encode('utf-8')).hexdigest()[:12]
        video_path = cache_dir / f'{rh}_video.pt'
        audio_path = cache_dir / f'{rh}_audio.pt'
        if video_path.exists() and audio_path.exists():
            continue
        with torch.no_grad():
            vf = preprocess_video_clip(rec['path'], train_mode=False).unsqueeze(0).to(device)
            af = preprocess_audio_clip(rec['path'], train_mode=False).to(device)
            video_features, audio_features = probe_model.extract_features(vf, af)
        torch.save(video_features.squeeze(0).detach().cpu(), str(video_path))
        torch.save(audio_features.squeeze(0).detach().cpu(), str(audio_path))


def build_speaker_stratified_splits(records: Sequence[Dict[str, str]]) -> Tuple[List[Dict[str, str]], List[Dict[str, str]], List[Dict[str, str]]]:
    speaker_to_records: Dict[str, List[Dict[str, str]]] = defaultdict(list)
    for record in records:
        speaker_to_records[record['speaker_id']].append(record)
    speaker_rows = []
    for speaker_id, speaker_records in speaker_to_records.items():
        dominant_label = Counter(item['label'] for item in speaker_records).most_common(1)[0][0]
        gender = speaker_records[0].get('gender', 'Unknown')
        speaker_rows.append({'speaker_id': speaker_id, 'label': dominant_label, 'gender': gender})
    frame = pd.DataFrame(speaker_rows)
    if frame.empty:
        return [], [], []

    n_speakers = len(frame)
    n_classes = len(CONFIG['class_names'])
    label_counts = frame['label'].value_counts()
    can_stratify = n_speakers >= 2 * n_classes and int(label_counts.min()) >= 2

    train_speakers = val_speakers = test_speakers = None

    if can_stratify:
        first_test_size = min(max(n_classes, n_speakers // 4), n_speakers - n_classes)
        if first_test_size >= n_classes and (n_speakers - first_test_size) >= n_classes:
            train_speakers, temp_speakers = train_test_split(
                frame,
                test_size=first_test_size,
                random_state=CONFIG['seed'],
                stratify=frame['label'],
            )
            temp_size = len(temp_speakers)
            if temp_size >= 2 * n_classes:
                val_speakers, test_speakers = train_test_split(
                    temp_speakers,
                    test_size=temp_size // 2,
                    random_state=CONFIG['seed'],
                    stratify=temp_speakers['label'],
                )
            else:
                temp_speakers = temp_speakers.sample(frac=1, random_state=CONFIG['seed']).reset_index(drop=True)
                mid = max(1, temp_size // 2)
                val_speakers = temp_speakers.iloc[:mid]
                test_speakers = temp_speakers.iloc[mid:]
        else:
            can_stratify = False

    if not can_stratify:
        shuffled = frame.sample(frac=1, random_state=CONFIG['seed']).reset_index(drop=True)
        n_train = max(1, int(round(n_speakers * 0.7)))
        n_val = max(1, int(round(n_speakers * 0.15)))
        if n_train + n_val >= n_speakers:
            n_val = max(1, n_speakers - n_train - 1)
        train_speakers = shuffled.iloc[:n_train]
        val_speakers = shuffled.iloc[n_train:n_train + n_val]
        test_speakers = shuffled.iloc[n_train + n_val:]

    train_ids = set(train_speakers['speaker_id'])
    val_ids = set(val_speakers['speaker_id'])
    test_ids = set(test_speakers['speaker_id'])
    train_records = [record for record in records if record['speaker_id'] in train_ids]
    val_records = [record for record in records if record['speaker_id'] in val_ids]
    test_records = [record for record in records if record['speaker_id'] in test_ids]

    test_genders = {rec.get('gender', 'Unknown') for rec in test_records}
    test_classes = {rec['label'] for rec in test_records}
    if 'Boys' not in test_genders or 'Girls' not in test_genders:
        print('WARNING: test split does not include both genders. Consider increasing test split size.')
    if len(test_classes) < len(CONFIG['class_names']):
        print('WARNING: test split does not include all classes. Consider increasing test split size.')

    print('Speaker split sizes (expected near 19/4/4 with 27 trainable speakers):', len(train_ids), len(val_ids), len(test_ids))
    return train_records, val_records, test_records


def build_class_balanced_sampler(records: Sequence[Dict[str, str]]) -> WeightedRandomSampler:
    if not records:
        raise ValueError('Cannot build a class-balanced sampler from an empty record list.')
    label_counts = Counter(record['label'] for record in records)
    weights = [1.0 / label_counts[record['label']] for record in records]
    return WeightedRandomSampler(weights=weights, num_samples=len(records), replacement=True)


def build_dataloaders(records: Sequence[Dict[str, str]]) -> Tuple[DataLoader, DataLoader, DataLoader]:
    train_records, val_records, test_records = build_speaker_stratified_splits(records)
    if CONFIG.get('use_feature_cache', False):
        cache_dir = WORK_DIR / 'feature_cache'
        precompute_and_cache_features(records, cache_dir, DEVICE)
        train_dataset = CachedDeepfakeDataset(train_records, cache_dir=cache_dir)
        val_dataset = CachedDeepfakeDataset(val_records, cache_dir=cache_dir)
        test_dataset = CachedDeepfakeDataset(test_records, cache_dir=cache_dir)
        collate_mode = 'cached'
    else:
        train_dataset = DeepfakeDataset(train_records, train_mode=True)
        val_dataset = DeepfakeDataset(val_records, train_mode=False)
        test_dataset = DeepfakeDataset(test_records, train_mode=False)
        collate_mode = 'raw'

    pin_mem = DEVICE.type == 'cuda'
    workers = CONFIG['num_workers']
    loader_kwargs = {
        'batch_size': CONFIG['batch_size'],
        'num_workers': workers,
        'pin_memory': pin_mem,
    }
    if workers > 0:
        loader_kwargs['persistent_workers'] = True
        loader_kwargs['prefetch_factor'] = 2

    train_loader = DataLoader(train_dataset, sampler=build_class_balanced_sampler(train_records), **loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)
    print('Dataloader mode:', collate_mode)
    return train_loader, val_loader, test_loader


print('Dataloaders ready')

## 12. Implement Training Loop with Mixed Precision

The training loop uses AMP and gradient accumulation for memory efficiency.

In [ ]:
def move_batch_to_device(batch: Dict[str, Any], device: torch.device) -> Dict[str, Any]:
    moved: Dict[str, Any] = {}
    for key, value in batch.items():
        if torch.is_tensor(value):
            moved[key] = value.to(device)
        else:
            moved[key] = value
    return moved


def train_one_epoch(
    model: MultimodalDeepfakeModel,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    criterion: nn.Module,
    device: torch.device,
    accumulation_steps: int = 1,
    stage_name: str = 'train',
    use_mixup: bool = False,
    ema: Optional[Any] = None,
) -> float:
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    total_batches = max(1, len(loader))

    for step, batch in enumerate(loader, start=1):
        batch = move_batch_to_device(batch, device)
        labels = batch['label']

        with autocast('cuda' if device.type == 'cuda' else 'cpu', enabled=CONFIG['use_amp']):
            if 'video_features' in batch and 'audio_features' in batch:
                video_features = batch['video_features']
                audio_features = batch['audio_features']
            else:
                video_features, audio_features = model.extract_features(batch['video_frames'], batch['audio_waveform'])

            video_aux_logits, audio_aux_logits = model.compute_aux_logits(video_features, audio_features)
            video_features, audio_features = model.apply_modality_dropout(video_features, audio_features)

            if use_mixup and random.random() < 0.5:
                lam = float(np.random.beta(CONFIG['mixup_alpha'], CONFIG['mixup_alpha']))
                perm = torch.randperm(labels.size(0), device=device)
                video_features = lam * video_features + (1.0 - lam) * video_features[perm]
                audio_features = lam * audio_features + (1.0 - lam) * audio_features[perm]
                labels_a, labels_b = labels, labels[perm]
                fusion_logits = model.fuse_logits(video_features, audio_features)
                if isinstance(fusion_logits, tuple):
                    fusion_logits = fusion_logits[0]
                fusion_loss = lam * criterion(fusion_logits, labels_a) + (1.0 - lam) * criterion(fusion_logits, labels_b)
            else:
                fusion_logits = model.fuse_logits(video_features, audio_features)
                if isinstance(fusion_logits, tuple):
                    fusion_logits = fusion_logits[0]
                fusion_loss = criterion(fusion_logits, labels)

            video_aux_loss = criterion(video_aux_logits, labels)
            audio_aux_loss = criterion(audio_aux_logits, labels)
            aux_w = float(CONFIG.get('aux_loss_weight', 0.3))
            loss = (fusion_loss + aux_w * video_aux_loss + aux_w * audio_aux_loss) / accumulation_steps

        scaler.scale(loss).backward()

        if step % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(model)

        running_loss += float(loss.item() * accumulation_steps)

        if step == 1 or step % max(1, total_batches // 5) == 0 or step == total_batches:
            print(f'[{stage_name}] batch {step}/{total_batches} loss={loss.item() * accumulation_steps:.4f}')

        if CONFIG.get('dry_run', False) and step >= 2:
            print('DRY RUN MODE: stopping epoch after 2 batches')
            break

    if total_batches % accumulation_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        if ema is not None:
            ema.update(model)

    effective_batches = min(total_batches, 2) if CONFIG.get('dry_run', False) else total_batches
    return running_loss / max(1, effective_batches)


print('Training loop ready')

## 13. Stage A: Train Unimodal Branches (Frozen Backbones)

This stage freezes the backbones and trains only the light adaptation layers first.

In [ ]:
def freeze_backbones(model: MultimodalDeepfakeModel) -> None:
    for parameter in model.video_backbone.parameters():
        parameter.requires_grad = False
    for parameter in model.audio_backbone.parameters():
        parameter.requires_grad = False


def freeze_early_layers(model: MultimodalDeepfakeModel) -> None:
    freeze_backbones(model)
    video_layers = getattr(getattr(model.video_backbone, 'model', None), 'layers', None)
    if video_layers is not None:
        for layer in video_layers:
            blocks = getattr(layer, 'blocks', None)
            if not blocks:
                continue
            freeze_count = max(1, len(blocks) // 2)
            for block in list(blocks)[:freeze_count]:
                for parameter in block.parameters():
                    parameter.requires_grad = False
    audio_encoder = getattr(getattr(model.audio_backbone, 'model', None), 'encoder', None)
    audio_layers = getattr(audio_encoder, 'layers', None)
    if audio_layers is not None:
        freeze_count = max(1, len(audio_layers) // 2)
        for layer in list(audio_layers)[:freeze_count]:
            for parameter in layer.parameters():
                parameter.requires_grad = False


def unfreeze_top_blocks(model: MultimodalDeepfakeModel) -> None:
    freeze_backbones(model)
    for parameter in model.fusion.parameters():
        parameter.requires_grad = True
    for parameter in model.classifier.parameters():
        parameter.requires_grad = True
    for parameter in model.attn_gate.parameters():
        parameter.requires_grad = True
    for parameter in model.video_aux_head.parameters():
        parameter.requires_grad = True
    for parameter in model.audio_aux_head.parameters():
        parameter.requires_grad = True


def _split_early_late(modules: Sequence[nn.Module]) -> Tuple[List[nn.Parameter], List[nn.Parameter]]:
    early: List[nn.Parameter] = []
    late: List[nn.Parameter] = []
    n = len(modules)
    mid = max(1, n // 2)
    for i, module in enumerate(modules):
        target = early if i < mid else late
        for p in module.parameters():
            if p.requires_grad:
                target.append(p)
    return early, late


def get_llrd_param_groups(model: MultimodalDeepfakeModel, base_lr: float, decay: float = 0.85) -> List[Dict[str, Any]]:
    param_groups: List[Dict[str, Any]] = []
    seen: set = set()

    def add_group(params: Sequence[nn.Parameter], lr_val: float) -> None:
        filtered = [p for p in params if p.requires_grad and id(p) not in seen]
        if filtered:
            for p in filtered:
                seen.add(id(p))
            param_groups.append({'params': filtered, 'lr': lr_val})

    video_blocks: List[nn.Module] = []
    for layer in getattr(model.video_backbone.model, 'layers', []):
        for block in getattr(layer, 'blocks', []):
            video_blocks.append(block)
    v_early, v_late = _split_early_late(video_blocks)
    add_group(v_early, base_lr * (decay ** 4))
    add_group(v_late, base_lr * (decay ** 2))

    audio_layers = list(getattr(getattr(model.audio_backbone.model, 'encoder', None), 'layers', []))
    a_early, a_late = _split_early_late(audio_layers)
    add_group(a_early, base_lr * (decay ** 4))
    add_group(a_late, base_lr * (decay ** 2))

    full_lr_modules = [
        model.video_backbone.temporal_proj,
        model.video_backbone.temporal_attn,
        model.video_backbone.temporal_ln,
        model.fusion,
        model.attn_gate,
        model.classifier,
        model.video_aux_head,
        model.audio_aux_head,
    ]
    full_params: List[nn.Parameter] = []
    for module in full_lr_modules:
        for p in module.parameters():
            if p.requires_grad:
                full_params.append(p)
    add_group(full_params, base_lr)

    remainder = [p for p in model.parameters() if p.requires_grad and id(p) not in seen]
    add_group(remainder, base_lr)
    return param_groups


class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = float(decay)
        self.shadow = {k: v.detach().float().cpu().clone() for k, v in model.state_dict().items()}

    def update(self, model: nn.Module) -> None:
        current = model.state_dict()
        for key, value in current.items():
            value_f = value.detach().float().cpu()
            if key not in self.shadow:
                self.shadow[key] = value_f.clone()
            else:
                self.shadow[key].mul_(self.decay).add_(value_f, alpha=1.0 - self.decay)

    def apply_to(self, model: nn.Module) -> None:
        model.load_state_dict(self.shadow, strict=False)

    def save(self, path: Path) -> None:
        torch.save(self.shadow, str(path))


def infonce_loss(video_features: torch.Tensor, audio_features: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    v = F.normalize(video_features, dim=1)
    a = F.normalize(audio_features, dim=1)
    sim = (v @ a.T) / temperature
    labels = torch.arange(sim.size(0), device=sim.device)
    return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels))


def stage_0_contrastive(model: MultimodalDeepfakeModel, train_loader: DataLoader, device: torch.device) -> None:
    print('--- Stage 0 contrastive start ---')
    for p in model.fusion.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = False
    for p in model.attn_gate.parameters():
        p.requires_grad = False
    for p in model.video_aux_head.parameters():
        p.requires_grad = False
    for p in model.audio_aux_head.parameters():
        p.requires_grad = False

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=CONFIG['weight_decay'])
    scaler = GradScaler(device='cuda' if device.type == 'cuda' else 'cpu', enabled=CONFIG['use_amp'])
    epochs = int(CONFIG.get('epochs_stage_0', 2))

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        step = 0
        for step, batch in enumerate(train_loader, start=1):
            batch = move_batch_to_device(batch, device)
            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda' if device.type == 'cuda' else 'cpu', enabled=CONFIG['use_amp']):
                if 'video_features' in batch and 'audio_features' in batch:
                    vf = batch['video_features']
                    af = batch['audio_features']
                else:
                    vf, af = model.extract_features(batch['video_frames'], batch['audio_waveform'])
                loss = infonce_loss(vf.mean(dim=1), af.mean(dim=1))
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += float(loss.item())
            if CONFIG.get('dry_run', False) and step >= 2:
                break
        print(f'[Stage 0] epoch {epoch}/{epochs} loss={running / max(1, step):.4f}')
    torch.save(model.state_dict(), str(WORK_DIR / 'Stage_0_contrastive.pth'))
    print('--- Stage 0 contrastive complete ---')


def evaluate_model(
    model: MultimodalDeepfakeModel,
    loader: DataLoader,
    criterion: Optional[nn.Module],
    device: torch.device,
    save_failure_cases: bool = True,
) -> Dict[str, float]:
    model.eval()
    eval_criterion = criterion if criterion is not None else nn.CrossEntropyLoss(label_smoothing=0.0)
    y_true: List[int] = []
    y_pred: List[int] = []
    y_prob: List[List[float]] = []
    losses: List[float] = []
    rows: List[Dict[str, Any]] = []

    with torch.no_grad():
        for step, batch in enumerate(loader, start=1):
            batch = move_batch_to_device(batch, device)
            labels = batch['label']
            if 'video_features' in batch and 'audio_features' in batch:
                logits = model.fuse_logits(batch['video_features'], batch['audio_features'])
                if isinstance(logits, tuple):
                    logits = logits[0]
            else:
                outputs = model(batch['video_frames'], batch['audio_waveform'])
                logits = outputs['fusion']
            loss = eval_criterion(logits, labels)
            losses.append(float(loss.item()))
            probs = torch.softmax(logits, dim=1)
            confs, preds = probs.max(dim=1)

            probs_cpu = probs.detach().cpu().numpy().tolist()
            preds_cpu = preds.detach().cpu().tolist()
            labels_cpu = labels.detach().cpu().tolist()
            confs_cpu = confs.detach().cpu().tolist()
            paths = batch.get('path', [''] * len(labels_cpu))

            y_prob.extend(probs_cpu)
            y_pred.extend(preds_cpu)
            y_true.extend(labels_cpu)

            for i in range(len(labels_cpu)):
                rows.append({
                    'path': paths[i],
                    'true_label': CONFIG['class_names'][int(labels_cpu[i])],
                    'pred_label': CONFIG['class_names'][int(preds_cpu[i])],
                    'confidence': float(confs_cpu[i]),
                    'probabilities': {CONFIG['class_names'][k]: float(probs_cpu[i][k]) for k in range(len(CONFIG['class_names']))},
                    'correct': bool(preds_cpu[i] == labels_cpu[i]),
                })

            if CONFIG.get('dry_run', False) and step >= 2:
                print('DRY RUN MODE: stopping evaluation after 2 batches')
                break

    auc = float('nan')
    if y_true and len(set(y_true)) > 1:
        try:
            auc = float(roc_auc_score(y_true, np.array(y_prob), multi_class='ovr', average='macro'))
        except ValueError as exc:
            print('WARNING: multiclass ROC AUC unavailable due to missing class in split:', exc)
            auc = float('nan')

    if save_failure_cases and rows:
        wrong = sorted([r for r in rows if not r['correct']], key=lambda x: x['confidence'], reverse=True)[:10]
        uncertain_right = sorted([r for r in rows if r['correct']], key=lambda x: x['confidence'])[:10]
        with (WORK_DIR / 'failure_cases.json').open('w', encoding='utf-8') as f:
            json.dump({'high_confidence_wrong': wrong, 'low_confidence_correct': uncertain_right}, f, indent=2)

    report = ''
    cm = []
    if y_true:
        labels_idx = list(range(len(CONFIG['class_names'])))
        cm = confusion_matrix(y_true, y_pred, labels=labels_idx).tolist()
        report = classification_report(y_true, y_pred, labels=labels_idx, target_names=CONFIG['class_names'], zero_division=0)

    return {
        'loss': float(np.mean(losses)) if losses else 0.0,
        'accuracy': float(accuracy_score(y_true, y_pred)) if y_true else 0.0,
        'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)) if y_true else 0.0,
        'recall_macro': float(recall_score(y_true, y_pred, average='macro', zero_division=0)) if y_true else 0.0,
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)) if y_true else 0.0,
        'roc_auc_macro_ovr': auc,
        'confusion_matrix': cm,
        'classification_report': report,
    }


def run_unimodal_baselines(records: Sequence[Dict[str, str]], device: torch.device) -> Dict[str, Dict[str, float]]:
    train_records, _, test_records = build_speaker_stratified_splits(records)
    train_ds = DeepfakeDataset(train_records, train_mode=False)
    test_ds = DeepfakeDataset(test_records, train_mode=False)
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'])
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])

    base_model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names'])).to(device)
    base_model.eval()
    freeze_backbones(base_model)

    video_probe = nn.Linear(base_model.video_backbone.feature_dim, len(CONFIG['class_names'])).to(device)
    audio_probe = nn.Linear(base_model.audio_backbone.feature_dim, len(CONFIG['class_names'])).to(device)

    v_opt = torch.optim.AdamW(video_probe.parameters(), lr=1e-3)
    a_opt = torch.optim.AdamW(audio_probe.parameters(), lr=1e-3)
    ce = nn.CrossEntropyLoss()

    for _epoch in range(5):
        video_probe.train()
        audio_probe.train()
        for batch in train_loader:
            batch = move_batch_to_device(batch, device)
            with torch.no_grad():
                vf, af = base_model.extract_features(batch['video_frames'], batch['audio_waveform'])
            vf = vf.mean(dim=1)
            af = af.mean(dim=1)

            v_opt.zero_grad(set_to_none=True)
            a_opt.zero_grad(set_to_none=True)
            v_loss = ce(video_probe(vf), batch['label'])
            a_loss = ce(audio_probe(af), batch['label'])
            v_loss.backward()
            a_loss.backward()
            v_opt.step()
            a_opt.step()
            if CONFIG.get('dry_run', False):
                break

    def eval_probe(which: str) -> Dict[str, float]:
        y_t, y_p = [], []
        for batch in test_loader:
            batch = move_batch_to_device(batch, device)
            with torch.no_grad():
                vf, af = base_model.extract_features(batch['video_frames'], batch['audio_waveform'])
                if which == 'video':
                    logits = video_probe(vf.mean(dim=1))
                else:
                    logits = audio_probe(af.mean(dim=1))
            preds = logits.argmax(dim=1)
            y_t.extend(batch['label'].detach().cpu().tolist())
            y_p.extend(preds.detach().cpu().tolist())
            if CONFIG.get('dry_run', False):
                break
        return {
            'accuracy': float(accuracy_score(y_t, y_p)) if y_t else 0.0,
            'f1_macro': float(f1_score(y_t, y_p, average='macro', zero_division=0)) if y_t else 0.0,
        }

    video_metrics = eval_probe('video')
    audio_metrics = eval_probe('audio')
    print('Unimodal baseline metrics:', {'video_only': video_metrics, 'audio_only': audio_metrics})

    for name, m in [('video', video_metrics), ('audio', audio_metrics)]:
        if m['accuracy'] > 0.95:
            print(f'WARNING: {name}-only probe exceeds 95% accuracy. Strong unimodal artifact suspected.')
        if m['accuracy'] < 0.60:
            print(f'WARNING: {name}-only probe below 60% accuracy. Preprocessing pipeline may be broken.')

    return {'video_only': video_metrics, 'audio_only': audio_metrics}


def bundle_outputs(files: Sequence[str], bundle_name: str) -> Path:
    import shutil
    tmpdir = WORK_DIR / (bundle_name + '_bundle')
    if tmpdir.exists():
        shutil.rmtree(tmpdir)
    tmpdir.mkdir(parents=True, exist_ok=True)
    for p in files:
        try:
            src = Path(p)
            if src.exists():
                shutil.copy2(str(src), str(tmpdir / src.name))
        except Exception as e:
            print('Warning copying', p, '->', e)
    archive_base = str(WORK_DIR / bundle_name)
    shutil.make_archive(archive_base, 'zip', root_dir=str(tmpdir))
    shutil.rmtree(tmpdir)
    archive_path = Path(archive_base + '.zip')
    print('Created bundle:', archive_path)
    return archive_path


def save_stage_checkpoint(
    stage_name: str,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    scheduler: Optional[Any],
    best_metrics: Dict[str, float],
    ema_state_dict: Optional[Dict[str, torch.Tensor]] = None,
) -> Path:
    checkpoint_path = WORK_DIR / f'{stage_name}_resume.pt'
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_metrics': best_metrics,
        'config': CONFIG,
    }
    if scheduler is not None:
        checkpoint['scheduler_state_dict'] = scheduler.state_dict()
    if ema_state_dict is not None:
        checkpoint['ema_state_dict'] = ema_state_dict
    torch.save(checkpoint, checkpoint_path)
    return checkpoint_path


def load_stage_checkpoint(
    stage_name: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    scheduler: Optional[Any],
    device: torch.device,
) -> Tuple[int, Dict[str, float], Optional[Dict[str, torch.Tensor]]]:
    checkpoint_path = WORK_DIR / f'{stage_name}_resume.pt'
    if not CONFIG['resume_from_checkpoints'] or not checkpoint_path.exists():
        return 1, {'f1_macro': -1.0}, None

    checkpoint = torch.load(str(checkpoint_path), map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    best_metrics = checkpoint.get('best_metrics', {'f1_macro': -1.0})
    ema_state_dict = checkpoint.get('ema_state_dict')
    start_epoch = int(checkpoint.get('epoch', 0)) + 1
    print(f'[{stage_name}] resumed from epoch {start_epoch - 1} using {checkpoint_path}')
    return start_epoch, best_metrics, ema_state_dict


def run_stage(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, stage_name: str, epochs: int, lr: float, device: torch.device) -> Dict[str, float]:
    train_records = getattr(getattr(train_loader, 'dataset', None), 'records', [])
    class_weights = build_class_weight_tensor(train_records, CONFIG['class_names']).to(device) if train_records else None
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    eval_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.0)

    param_groups = get_llrd_param_groups(model, base_lr=lr, decay=0.85)
    optimizer = torch.optim.AdamW(param_groups, weight_decay=CONFIG['weight_decay'])
    warmup_epochs = max(1, epochs // 10)

    def lr_lambda(epoch_idx: int) -> float:
        if epoch_idx < warmup_epochs:
            return (epoch_idx + 1) / warmup_epochs
        progress = (epoch_idx - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = GradScaler(device='cuda' if device.type == 'cuda' else 'cpu', enabled=CONFIG['use_amp'])

    start_epoch, best_metrics, ema_state = load_stage_checkpoint(stage_name, model, optimizer, scaler, scheduler, device)
    ema = ModelEMA(model, decay=float(CONFIG.get('ema_decay', 0.999))) if (stage_name == 'Stage D' and CONFIG.get('use_ema', False)) else None
    if ema is not None and ema_state is not None:
        ema.shadow = {k: v.detach().float().cpu().clone() for k, v in ema_state.items()}

    print(f'--- {stage_name} start ---')
    if class_weights is not None:
        print(f'[{stage_name}] class weights:', class_weights.detach().cpu().tolist())

    use_mixup = stage_name in {'Stage C', 'Stage D'}
    for epoch in range(start_epoch, epochs + 1):
        print(f'[{stage_name}] epoch {epoch}/{epochs} started')
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
            criterion,
            device,
            CONFIG['accumulation_steps'],
            stage_name=stage_name,
            use_mixup=use_mixup,
            ema=ema,
        )
        val_metrics = evaluate_model(model, val_loader, eval_criterion, device, save_failure_cases=False)
        scheduler.step()
        print(f"[{stage_name}] epoch={epoch} train_loss={train_loss:.4f} val_f1={val_metrics['f1_macro']:.4f} val_acc={val_metrics['accuracy']:.4f}")

        if CONFIG['checkpoint_every_epoch']:
            save_stage_checkpoint(stage_name, epoch, model, optimizer, scaler, scheduler, best_metrics, ema_state_dict=ema.shadow if ema is not None else None)

        if val_metrics['f1_macro'] > best_metrics['f1_macro']:
            best_metrics = val_metrics
            save_stage_checkpoint(stage_name, epoch, model, optimizer, scaler, scheduler, best_metrics, ema_state_dict=ema.shadow if ema is not None else None)
            best_ckpt = WORK_DIR / f"{stage_name}_best.pth"
            torch.save(model.state_dict(), str(best_ckpt))

    final_ckpt = WORK_DIR / f"{stage_name}_final.pth"
    torch.save(model.state_dict(), str(final_ckpt))

    if ema is not None:
        ema_path = WORK_DIR / 'Stage_D_ema.pth'
        ema.save(ema_path)
        print('Saved EMA checkpoint:', ema_path)

    metrics_path = WORK_DIR / f"{stage_name}_metrics.json"
    with metrics_path.open('w', encoding='utf-8') as f:
        json.dump(best_metrics, f, indent=2)

    print(f'--- {stage_name} complete ---')
    return best_metrics


def stage_a_train(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    freeze_backbones(model)
    return run_stage(model, train_loader, val_loader, 'Stage A', CONFIG['epochs_stage_a'], CONFIG['lr_stage_a'], device)


print('Stage A ready')

## 14. Stage B: Fine-tune Branch Top Blocks

This stage unfreezes the final parts of the model and lowers the learning rate.

In [ ]:
def stage_b_finetune(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    unfreeze_top_blocks(model)
    return run_stage(model, train_loader, val_loader, 'Stage B', CONFIG['epochs_stage_b'], CONFIG['lr_stage_b'], device)


print('Stage B ready')

## 15. Stage C: Train Fusion with Partial Freezing

The fusion stage emphasizes cross-attention while keeping the backbones mostly fixed.

In [ ]:
def stage_c_train_fusion(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    freeze_early_layers(model)
    return run_stage(model, train_loader, val_loader, 'Stage C', CONFIG['epochs_stage_c'], CONFIG['lr_stage_c'], device)


print('Stage C ready')

## 16. Stage D: End-to-End Low Learning-Rate Fine-tuning

This is the final polishing step after the staged pretraining and fusion steps.

In [ ]:
def stage_d_end_to_end(model: MultimodalDeepfakeModel, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    unfreeze_top_blocks(model)
    return run_stage(model, train_loader, val_loader, 'Stage D', CONFIG['epochs_stage_d'], CONFIG['lr_stage_d'], device)


print('Stage D ready')

## 17. Evaluation and Metrics

The evaluation block computes the core classification metrics and confusion matrix.

In [ ]:
def evaluate_and_save(
    model: MultimodalDeepfakeModel,
    test_loader: DataLoader,
    device: torch.device,
    output_name: str = 'pilot_metrics.json',
) -> Dict[str, object]:
    metrics: Dict[str, object] = dict(evaluate_model(model, test_loader, criterion=None, device=device, save_failure_cases=True))

    language_stats: Dict[str, Dict[str, float]] = {}
    language_buckets: Dict[str, Dict[str, List[int]]] = {
        'Bangla': {'y_true': [], 'y_pred': []},
        'English': {'y_true': [], 'y_pred': []},
    }

    model.eval()
    with torch.no_grad():
        for step, batch in enumerate(test_loader, start=1):
            batch = move_batch_to_device(batch, device)
            labels = batch['label']
            if 'video_features' in batch and 'audio_features' in batch:
                logits = model.fuse_logits(batch['video_features'], batch['audio_features'])
                if isinstance(logits, tuple):
                    logits = logits[0]
            else:
                logits = model(batch['video_frames'], batch['audio_waveform'])['fusion']
            predictions = logits.argmax(dim=1).detach().cpu().tolist()
            labels_cpu = labels.detach().cpu().tolist()
            paths = batch.get('path', [])
            for index, path in enumerate(paths):
                language = 'Bangla' if 'Bangla' in path else 'English' if 'English' in path else None
                if language is None:
                    continue
                language_buckets[language]['y_true'].append(int(labels_cpu[index]))
                language_buckets[language]['y_pred'].append(int(predictions[index]))
            if CONFIG.get('dry_run', False) and step >= 2:
                break

    for language, bucket in language_buckets.items():
        if bucket['y_true']:
            language_stats[language] = {
                'accuracy': float(accuracy_score(bucket['y_true'], bucket['y_pred'])),
                'f1_macro': float(f1_score(bucket['y_true'], bucket['y_pred'], average='macro', zero_division=0)),
            }
        else:
            language_stats[language] = {'accuracy': 0.0, 'f1_macro': 0.0}

    metrics['language_breakdown'] = language_stats
    metrics_path = WORK_DIR / output_name
    with metrics_path.open('w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2)
    print('Saved metrics to', metrics_path)
    return metrics


def evaluate_modality_ablations(model: MultimodalDeepfakeModel, test_loader: DataLoader, device: torch.device) -> Dict[str, Dict[str, float]]:
    model.eval()
    results: Dict[str, Dict[str, float]] = {}

    def collect(mode: str) -> Dict[str, float]:
        y_t: List[int] = []
        y_p: List[int] = []
        with torch.no_grad():
            for step, batch in enumerate(test_loader, start=1):
                batch = move_batch_to_device(batch, device)
                if 'video_features' in batch and 'audio_features' in batch:
                    vf, af = batch['video_features'], batch['audio_features']
                else:
                    vf, af = model.extract_features(batch['video_frames'], batch['audio_waveform'])

                if mode == 'full':
                    logits = model.fuse_logits(vf, af)
                    if isinstance(logits, tuple):
                        logits = logits[0]
                elif mode == 'video_only':
                    logits = model.video_aux_head(vf.mean(dim=1))
                else:
                    logits = model.audio_aux_head(af.mean(dim=1))

                preds = logits.argmax(dim=1)
                y_t.extend(batch['label'].detach().cpu().tolist())
                y_p.extend(preds.detach().cpu().tolist())
                if CONFIG.get('dry_run', False) and step >= 2:
                    break
        return {
            'accuracy': float(accuracy_score(y_t, y_p)) if y_t else 0.0,
            'f1_macro': float(f1_score(y_t, y_p, average='macro', zero_division=0)) if y_t else 0.0,
        }

    results['full_multimodal'] = collect('full')
    results['video_only'] = collect('video_only')
    results['audio_only'] = collect('audio_only')

    print('Modality ablation results:')
    print(f"  full_multimodal acc={results['full_multimodal']['accuracy']:.4f} f1={results['full_multimodal']['f1_macro']:.4f}")
    print(f"  video_only      acc={results['video_only']['accuracy']:.4f} f1={results['video_only']['f1_macro']:.4f}")
    print(f"  audio_only      acc={results['audio_only']['accuracy']:.4f} f1={results['audio_only']['f1_macro']:.4f}")

    best_uni = max(results['video_only']['accuracy'], results['audio_only']['accuracy'])
    if results['full_multimodal']['accuracy'] < best_uni + 0.03:
        print('WARNING: Cross-attention fusion is not outperforming unimodal baselines — check that sequence dimensions are preserved before fusion.')

    return results


def compute_calibration(model: MultimodalDeepfakeModel, test_loader: DataLoader, device: torch.device) -> Dict[str, Any]:
    model.eval()
    all_probs: List[List[float]] = []
    all_labels: List[int] = []

    with torch.no_grad():
        for step, batch in enumerate(test_loader, start=1):
            batch = move_batch_to_device(batch, device)
            if 'video_features' in batch and 'audio_features' in batch:
                logits = model.fuse_logits(batch['video_features'], batch['audio_features'])
                if isinstance(logits, tuple):
                    logits = logits[0]
            else:
                logits = model(batch['video_frames'], batch['audio_waveform'])['fusion']
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(batch['label'].detach().cpu().tolist())
            if CONFIG.get('dry_run', False) and step >= 2:
                break

    probs_np = np.array(all_probs) if all_probs else np.zeros((0, len(CONFIG['class_names'])))
    labels_np = np.array(all_labels) if all_labels else np.zeros((0,), dtype=np.int64)

    calibration_summary: Dict[str, Any] = {'per_class_ece': {}, 'overall_ece': float('nan')}
    all_bin_weights: List[float] = []
    all_bin_errors: List[float] = []

    for class_idx, class_name in enumerate(CONFIG['class_names']):
        if probs_np.shape[0] == 0:
            continue
        y_true = (labels_np == class_idx).astype(np.int32)
        y_prob = probs_np[:, class_idx]
        frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy='uniform')

        # ECE
        bins = np.linspace(0.0, 1.0, 11)
        ece = 0.0
        total = len(y_true)
        for b in range(10):
            left, right = bins[b], bins[b + 1]
            mask = (y_prob >= left) & (y_prob < right if b < 9 else y_prob <= right)
            if not np.any(mask):
                continue
            acc_bin = float(np.mean(y_true[mask]))
            conf_bin = float(np.mean(y_prob[mask]))
            w = float(np.sum(mask)) / max(1, total)
            ece += w * abs(acc_bin - conf_bin)
            all_bin_weights.append(w)
            all_bin_errors.append(abs(acc_bin - conf_bin))

        calibration_summary['per_class_ece'][class_name] = float(ece)
        with (WORK_DIR / f'calibration_{class_name}.json').open('w', encoding='utf-8') as f:
            json.dump({'fraction_of_positives': frac_pos.tolist(), 'mean_predicted_value': mean_pred.tolist(), 'ece': float(ece)}, f, indent=2)
        print(f'Class {class_name} ECE: {ece:.4f}')

    if all_bin_weights:
        overall_ece = float(np.sum(np.array(all_bin_weights) * np.array(all_bin_errors)) / max(1e-12, np.sum(all_bin_weights)))
    else:
        overall_ece = float('nan')
    calibration_summary['overall_ece'] = overall_ece
    print(f'Overall ECE: {overall_ece:.4f}')
    return calibration_summary


def log_attention_weights(model: MultimodalDeepfakeModel, test_loader: DataLoader, device: torch.device, n_samples: int = 20) -> Dict[str, str]:
    model.eval()
    model.fusion.return_weights = True

    per_class_video: Dict[str, List[torch.Tensor]] = defaultdict(list)
    per_class_audio: Dict[str, List[torch.Tensor]] = defaultdict(list)
    seen = 0

    with torch.no_grad():
        for batch in test_loader:
            batch = move_batch_to_device(batch, device)
            if 'video_features' in batch and 'audio_features' in batch:
                fusion_out = model.fuse_logits(batch['video_features'], batch['audio_features'])
                if isinstance(fusion_out, tuple) and len(fusion_out) == 3:
                    logits, video_w, audio_w = fusion_out
                else:
                    continue
            else:
                outputs = model(batch['video_frames'], batch['audio_waveform'])
                logits = outputs['fusion']
                video_w = outputs.get('video_attn_weights')
                audio_w = outputs.get('audio_attn_weights')
                if video_w is None or audio_w is None:
                    continue

            labels = batch['label'].detach().cpu().tolist()
            if video_w.dim() == 4:
                video_w = video_w.mean(dim=1)
            if audio_w.dim() == 4:
                audio_w = audio_w.mean(dim=1)

            for i, label_idx in enumerate(labels):
                class_name = CONFIG['class_names'][int(label_idx)]
                per_class_video[class_name].append(video_w[i].detach().cpu())
                per_class_audio[class_name].append(audio_w[i].detach().cpu())
                seen += 1
                if seen >= n_samples:
                    break
            if seen >= n_samples:
                break

    model.fusion.return_weights = False

    saved: Dict[str, str] = {}
    for class_name in CONFIG['class_names']:
        if not per_class_video[class_name]:
            continue
        video_mean = torch.stack(per_class_video[class_name]).mean(dim=0)
        audio_mean = torch.stack(per_class_audio[class_name]).mean(dim=0)
        v_path = WORK_DIR / f'attention_weights_{class_name}_video.pt'
        a_path = WORK_DIR / f'attention_weights_{class_name}_audio.pt'
        torch.save(video_mean, str(v_path))
        torch.save(audio_mean, str(a_path))
        saved[class_name] = f'{v_path.name}, {a_path.name}'

        # report most attended positions
        v_focus = int(video_mean.mean(dim=0).argmax().item()) if video_mean.numel() else -1
        a_focus = int(audio_mean.mean(dim=0).argmax().item()) if audio_mean.numel() else -1
        print(f'{class_name}: video->audio peak timestep {v_focus}, audio->video peak frame {a_focus}')

    return saved


print('Evaluation ready')

## 18. Scope for Full Dataset Training

When the pilot is stable, switch `FULL_DATASET_MODE = True` and keep the same staged training plan for the entire corpus.

Scale-up notes:
- keep speaker-aware splitting
- keep class-balanced sampling
- keep mixed precision and gradient accumulation
- save checkpoints after each stage
- use low learning rates for the final fine-tune

In [ ]:
def run_pilot_training_pipeline() -> Optional[Dict[str, object]]:
    required_names = [
        'run_unimodal_baselines',
        'stage_a_train',
        'stage_b_finetune',
        'stage_c_train_fusion',
        'stage_d_end_to_end',
        'evaluate_and_save',
        'evaluate_modality_ablations',
        'compute_calibration',
        'log_attention_weights',
        'build_dataloaders',
        'MultimodalDeepfakeModel',
    ]
    missing_names = [name for name in required_names if name not in globals()]
    if missing_names:
        print('Training setup is not ready yet.')
        print('Run the earlier notebook cells first, then rerun this cell.')
        print('Missing names:', ', '.join(missing_names))
        return None

    pilot_train_loader, pilot_val_loader, pilot_test_loader = build_dataloaders(pilot_records)
    pilot_model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names'])).to(DEVICE)
    print('Pilot loaders built:', len(pilot_train_loader), len(pilot_val_loader), len(pilot_test_loader))

    baseline_metrics = run_unimodal_baselines(pilot_records, DEVICE)
    print('Baseline check complete.')

    if RUN_TRAINING:
        original_epochs = {
            'epochs_stage_0': CONFIG['epochs_stage_0'],
            'epochs_stage_a': CONFIG['epochs_stage_a'],
            'epochs_stage_b': CONFIG['epochs_stage_b'],
            'epochs_stage_c': CONFIG['epochs_stage_c'],
            'epochs_stage_d': CONFIG['epochs_stage_d'],
        }
        if CONFIG.get('dry_run', False):
            print('DRY RUN MODE — 2 batches per stage')
            CONFIG['epochs_stage_0'] = 1
            CONFIG['epochs_stage_a'] = 1
            CONFIG['epochs_stage_b'] = 1
            CONFIG['epochs_stage_c'] = 1
            CONFIG['epochs_stage_d'] = 1

        if CONFIG.get('run_stage_0', False):
            stage_0_contrastive(pilot_model, pilot_train_loader, DEVICE)

        _stage_a_metrics = stage_a_train(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)
        _stage_b_metrics = stage_b_finetune(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)
        _stage_c_metrics = stage_c_train_fusion(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)
        _stage_d_metrics = stage_d_end_to_end(pilot_model, pilot_train_loader, pilot_val_loader, DEVICE)

        # Final pilot evaluation using EMA if enabled
        eval_model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names'])).to(DEVICE)
        eval_model.load_state_dict(pilot_model.state_dict(), strict=False)
        if CONFIG.get('use_ema', False):
            ema_path = WORK_DIR / 'Stage_D_ema.pth'
            if ema_path.exists():
                eval_model.load_state_dict(torch.load(str(ema_path), map_location=DEVICE), strict=False)
                print('Loaded EMA weights for final evaluation:', ema_path)

        final_metrics = evaluate_and_save(eval_model, pilot_test_loader, DEVICE, output_name='pilot_metrics.json')
        modality_ablation = evaluate_modality_ablations(eval_model, pilot_test_loader, DEVICE)
        calibration = compute_calibration(eval_model, pilot_test_loader, DEVICE)
        attention_logs = log_attention_weights(eval_model, pilot_test_loader, DEVICE, n_samples=20)

        # Held-out evaluation exactly once after Stage D
        held_out_loader = DataLoader(
            DeepfakeDataset(held_out_records, train_mode=False),
            batch_size=CONFIG['batch_size'],
            shuffle=False,
            num_workers=CONFIG['num_workers'],
        )
        held_out_metrics = evaluate_and_save(eval_model, held_out_loader, DEVICE, output_name='held_out_metrics.json')
        print('Held-out evaluation complete:', held_out_metrics.get('accuracy', 0.0))

        # append post-hoc analysis to pilot metrics file
        pilot_metrics_path = WORK_DIR / 'pilot_metrics.json'
        if pilot_metrics_path.exists():
            with pilot_metrics_path.open('r', encoding='utf-8') as f:
                pilot_payload = json.load(f)
            pilot_payload['baseline_metrics'] = baseline_metrics
            pilot_payload['modality_ablation'] = modality_ablation
            pilot_payload['calibration'] = calibration
            pilot_payload['attention_logs'] = attention_logs
            with pilot_metrics_path.open('w', encoding='utf-8') as f:
                json.dump(pilot_payload, f, indent=2)

        for k, v in original_epochs.items():
            CONFIG[k] = v

        print(final_metrics)
        return final_metrics

    print('Set RUN_TRAINING = True in cell 3 to execute the pilot training pipeline.')

    def bundle_downloadable_outputs(work_dir: Path) -> List[Path]:
        bundles: List[Path] = []
        output_groups = {
            'pilot_manifest': [work_dir / 'pilot_manifest_per_speaker_4.json'],
            'pilot_metrics': [work_dir / 'pilot_metrics.json'],
            'held_out_metrics': [work_dir / 'held_out_metrics.json'],
        }
        for bundle_name, files in output_groups.items():
            bundle_path = work_dir / f'{bundle_name}.zip'
            with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
                for file_path in files:
                    if file_path.exists():
                        archive.write(file_path, arcname=file_path.name)
            bundles.append(bundle_path)
        return bundles

    def display_download_links(paths: Sequence[Path]) -> None:
        for path in paths:
            if path.exists():
                print('Download:', path)
                try:
                    display(FileLink(str(path)))
                except Exception:
                    pass

    bundle_paths = bundle_downloadable_outputs(WORK_DIR)
    display_download_links(bundle_paths)
    print('Downloadable artifacts prepared:', [path.name for path in bundle_paths])
    return None


run_pilot_training_pipeline()

## Downloadable Outputs

The notebook writes pilot artifacts into the Kaggle working directory and bundles them as zip files for download at the end of the run. If you execute the training stages, the final cell will generate clickable links for the manifest and metrics bundles.

## 19. Test a Single Video After Cross-Domain Fusion

Set the video path and checkpoint path below to run fused video+audio inference on one clip.

In [ ]:
# Provide your test video path here (Kaggle example: /kaggle/input/<dataset>/<subdir>/clip.mp4)
TEST_VIDEO_PATH = ''

# Use EMA checkpoint by default when enabled, otherwise Stage D, then Stage C
CHECKPOINT_CANDIDATES = [
    WORK_DIR / 'Stage_D_ema.pth',
    WORK_DIR / 'Stage D_final.pth',
    WORK_DIR / 'Stage D_best.pth',
    WORK_DIR / 'Stage C_final.pth',
    WORK_DIR / 'Stage C_best.pth',
]


def get_best_checkpoint_path(candidates: Sequence[Path]) -> Optional[Path]:
    for path in candidates:
        if path.exists():
            return path
    return None


def load_model_for_inference(device: torch.device, checkpoint_path: Optional[Path] = None) -> MultimodalDeepfakeModel:
    inference_model = MultimodalDeepfakeModel(num_classes=len(CONFIG['class_names'])).to(device)
    if checkpoint_path is not None and checkpoint_path.exists():
        state = torch.load(str(checkpoint_path), map_location=device)
        inference_model.load_state_dict(state, strict=False)
        print('Loaded checkpoint:', checkpoint_path)
    else:
        print('No checkpoint found. Using current model weights.')
    inference_model.eval()
    return inference_model


def _build_waveform_variant(video_path: str, offset_seconds: float = 0.0, noise_std: float = 0.0) -> torch.Tensor:
    waveform = preprocess_audio_clip(video_path, train_mode=False)
    if offset_seconds > 0:
        shift = int(offset_seconds * CONFIG['audio_sample_rate'])
        if shift > 0 and shift < waveform.size(1):
            waveform = torch.cat([waveform[:, shift:], torch.zeros_like(waveform[:, :shift])], dim=1)
    if noise_std > 0:
        waveform = waveform + noise_std * torch.randn_like(waveform)
    return waveform


@torch.no_grad()
def predict_single_video(video_path: str, model: MultimodalDeepfakeModel, device: torch.device, use_tta: bool = True) -> Dict[str, object]:
    base_video = preprocess_video_clip(video_path, train_mode=False).unsqueeze(0).to(device)
    base_audio = preprocess_audio_clip(video_path, train_mode=False).to(device)

    def run_pass(video_tensor: torch.Tensor, audio_tensor: torch.Tensor) -> np.ndarray:
        out = model(video_tensor, audio_tensor)
        logits = out['fusion']
        return torch.softmax(logits, dim=1).squeeze(0).detach().cpu().numpy()

    probs_list: List[np.ndarray] = []
    probs_list.append(run_pass(base_video, base_audio))

    if use_tta:
        # Pass 2: horizontally flipped frames
        probs_list.append(run_pass(torch.flip(base_video, dims=[4]), base_audio))

        # Pass 3: +5% brightness
        bright_video = torch.clamp(base_video * 1.05, min=base_video.min().item(), max=base_video.max().item())
        probs_list.append(run_pass(bright_video, base_audio))

        # Pass 4: audio offset +0.5s
        offset_audio = _build_waveform_variant(video_path, offset_seconds=0.5, noise_std=0.0).to(device)
        probs_list.append(run_pass(base_video, offset_audio))

        # Pass 5: audio Gaussian noise
        noisy_audio = _build_waveform_variant(video_path, offset_seconds=0.0, noise_std=0.005).to(device)
        probs_list.append(run_pass(base_video, noisy_audio))

    mean_probs = np.mean(np.stack(probs_list, axis=0), axis=0)
    pred_idx = int(np.argmax(mean_probs))
    pred_label = CONFIG['class_names'][pred_idx]
    prob_map = {CONFIG['class_names'][i]: float(mean_probs[i]) for i in range(len(CONFIG['class_names']))}
    return {
        'predicted_class': pred_label,
        'confidence': float(mean_probs[pred_idx]),
        'probabilities': prob_map,
    }


if TEST_VIDEO_PATH.strip():
    test_path = Path(TEST_VIDEO_PATH)
    if not test_path.exists():
        print('Video not found:', test_path)
    else:
        best_ckpt = get_best_checkpoint_path(CHECKPOINT_CANDIDATES)
        infer_model = load_model_for_inference(DEVICE, best_ckpt)
        result = predict_single_video(str(test_path), infer_model, DEVICE, use_tta=True)
        print('Prediction:', result['predicted_class'])
        print('Confidence:', round(float(result['confidence']), 4))
        print('Class probabilities:', json.dumps(result['probabilities'], indent=2))
else:
    print('Set TEST_VIDEO_PATH to run single-video inference after fusion training.')